In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

PROC = Path("../data/processed")

counts = pd.read_csv(PROC / "BRCA_pam50_counts.csv", index_col="gene_id")
meta   = pd.read_csv(PROC / "BRCA_pam50_meta.csv")
genes  = pd.read_csv(PROC / "TCGA_BRCA_gene_info.csv", index_col="gene_id")
assert (counts.columns == meta["sample_id"]).all()       # same sample order
print("Start:", counts.shape)

# 1. Keep protein-coding genes 
pc = genes.index[genes["gene_type"] == "protein_coding"]
counts = counts.loc[counts.index.intersection(pc)]
print("Protein-coding:", counts.shape)

# 2. Library size (sequencing depth per sample, millions of reads)
lib = counts.sum(axis=0)
print((lib / 1e6).describe().round(2))

# 3. Remove low-expression genes
# keep a gene if CPM >= 1 in a least as many samples as the smallest subtype (40)
min_samples = meta["subtype"].value_counts().min()
cpm  = counts / lib * 1e6
keep = (cpm >= 1).sum(axis=1) >= min_samples
counts = counts[keep]
print(f"Expressed genes (CPM>=1 in >= {min_samples} samples):", counts.shape)

# 4. Normalise: log2(CPM + 1)
logcpm = np.log2(counts / counts.sum(axis=0) * 1e6 + 1)

# use gene symbols (ESR1, ERBB2...) instead of Ensembl IDs
sym = genes.loc[logcpm.index, "gene_name"]
order = logcpm.mean(axis=1).sort_values(ascending=False).index
keep_ids = sym.loc[order].drop_duplicates().index      # if a symbol repeats, keep the most expressed
logcpm = logcpm.loc[keep_ids].rename(index=sym).sort_index()
logcpm.index.name = "gene"
print("log2-CPM matrix (genes x samples):", logcpm.shape)

#  5. Clean clinical variables 
def stage_group(s):
    if pd.isna(s) or s in ("Stage X", "[Discrepancy]"):
        return np.nan
    s = s.replace("Stage ", "")
    for g in ("IV", "III", "II", "I"):          # check IV before I, III before II
        if s.startswith(g):
            return "Stage " + g
    return np.nan

meta["stage_group"] = meta["stage"].apply(stage_group)
meta["age"] = pd.to_numeric(meta["age"], errors="coerce")
for col in ["ER", "PR"]:
    meta[col] = meta[col].where(meta[col].isin(["Positive", "Negative"]))
meta["HER2_IHC"] = meta["HER2_IHC"].where(meta["HER2_IHC"].isin(["Positive", "Negative", "Equivocal"]))
meta["library_size_M"] = (lib / 1e6).values
print(meta["stage_group"].value_counts(dropna=False))


counts.to_csv(PROC / "counts_filtered.csv")     # raw counts  -> differential expression (DESeq2)
logcpm.to_csv(PROC / "logcpm.csv")              # log2-CPM    -> EDA, clustering, ML
meta.to_csv(PROC / "meta_clean.csv", index=False)
print("Saved counts_filtered.csv, logcpm.csv, meta_clean.csv")

Start: (60660, 1083)
Protein-coding: (19962, 1083)
count    1083.00
mean       56.45
std        14.11
min        18.92
25%        46.44
50%        55.91
75%        66.46
max       110.90
dtype: float64
Expressed genes (CPM>=1 in >= 40 samples): (15959, 1083)
log2-CPM matrix (genes x samples): (15959, 1083)
stage_group
Stage II     612
Stage III    247
Stage I      182
NaN           23
Stage IV      19
Name: count, dtype: int64
Saved counts_filtered.csv, logcpm.csv, meta_clean.csv
